# Python drills 8: concurrency

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/python/practice/python-drills-08-concurrency.ipynb)

Companion to the [Concurrency](../learning/python-08-concurrency.ipynb) chapter. The exercises use thread pools, locks, queues and `asyncio`. Each async check is awaited at the top level of its cell, which notebooks support.

Try each exercise in the code cell below its question. Use the sample inputs to compare your result with the expected output. Expand **Show solution** when you want to review the approach.

## Exercise 1: map in threads, keeping the input order

**Difficulty:** Medium · about 10 min

Write `parallel_map(fn, items)`. Run `fn` on every item using a `ThreadPoolExecutor` with four workers, and return the results as a list in the same order as `items`.

**Sample inputs and expected outputs**

| Call | Expected output |
| --- | --- |
| `parallel_map(lambda x: x * x, [1, 2, 3])` | `[1, 4, 9]` |
| `parallel_map(str.upper, ['a', 'b'])` | `['A', 'B']` |
| `parallel_map(str, [])` | `[]` |

In [ ]:
from concurrent.futures import ThreadPoolExecutor

def parallel_map(fn, items):
    # your code here
    pass

<details>
<summary><b>Show solution</b></summary>

```python
from concurrent.futures import ThreadPoolExecutor

def parallel_map(fn, items):
    with ThreadPoolExecutor(max_workers=4) as pool:
        return list(pool.map(fn, items))
```

</details>

## Exercise 2: a counter that threads update safely

**Difficulty:** Medium · about 10 min

Write `count_with_lock(workers, increments)`. Start `workers` threads. Each thread adds 1 to a shared counter `increments` times, and holds a `threading.Lock` for each addition. Return the final total.

**Sample inputs and expected outputs**

| Call | Expected output |
| --- | --- |
| `count_with_lock(4, 1000)` | `4000` |
| `count_with_lock(1, 5)` | `5` |
| `count_with_lock(3, 0)` | `0` |

In [ ]:
import threading

def count_with_lock(workers, increments):
    # your code here
    pass

<details>
<summary><b>Show solution</b></summary>

```python
import threading

def count_with_lock(workers, increments):
    lock = threading.Lock()
    total = [0]

    def bump():
        for _ in range(increments):
            with lock:
                total[0] += 1

    threads = [threading.Thread(target=bump) for _ in range(workers)]
    for thread in threads:
        thread.start()
    for thread in threads:
        thread.join()
    return total[0]
```

</details>

## Exercise 3: sum numbers with a queue and worker threads

**Difficulty:** Medium · about 10 min

Write `sum_with_queue(numbers, workers)`. Put every number on a `queue.Queue`. Start `workers` threads that take numbers off the queue and store them in a shared list, under a lock. Stop each worker with a `None` sentinel. Return the sum of the stored numbers. An empty input gives `0`.

**Sample inputs and expected outputs**

| Call | Expected output |
| --- | --- |
| `sum_with_queue(range(10), 3)` | `45` |
| `sum_with_queue([], 2)` | `0` |
| `sum_with_queue([7], 2)` | `7` |


*Hint: put one `None` on the queue for each worker, after all the numbers.* Hold a lock around each append, as a synchronization exercise; CPython list.append is already atomic, so the lock is not necessary for this particular list operation.  in append.

In [ ]:
import queue
import threading

def sum_with_queue(numbers, workers):
    # your code here
    pass

<details>
<summary><b>Show solution</b></summary>

```python
import queue
import threading

def sum_with_queue(numbers, workers):
    tasks = queue.Queue()
    stored = []
    lock = threading.Lock()

    def worker():
        while True:
            item = tasks.get()
            if item is None:
                return
            with lock:
                stored.append(item)

    threads = [threading.Thread(target=worker) for _ in range(workers)]
    for thread in threads:
        thread.start()
    for number in numbers:
        tasks.put(number)
    for _ in threads:
        tasks.put(None)
    for thread in threads:
        thread.join()
    return sum(stored)
```

</details>

## Exercise 4: run coroutines with a concurrency limit

**Difficulty:** Medium · about 10 min

Write `async def gather_limited(fn, items, limit)`. `fn` is an async function. Run `fn(item)` for every item at the same time, but with at most `limit` calls in progress at once. Return the results in the input order.

**Sample input and expected output**

```python
async def double(value):
    await asyncio.sleep(0.01)
    return value * 2

await gather_limited(double, [3, 1, 2], 2)
# Expected: [6, 2, 4]; at most two calls may run at once
await gather_limited(double, [], 2)
# Expected: []
```


*Hint: an `asyncio.Semaphore` with `async with` limits the calls, and `asyncio.gather` keeps the order.*

In [ ]:
import asyncio

async def gather_limited(fn, items, limit):
    # your code here
    pass

<details>
<summary><b>Show solution</b></summary>

```python
import asyncio

async def gather_limited(fn, items, limit):
    gate = asyncio.Semaphore(limit)

    async def run(item):
        async with gate:
            return await fn(item)

    return await asyncio.gather(*(run(item) for item in items))
```

</details>

## Exercise 5: keep errors as results

**Difficulty:** Medium · about 10 min

`divide` is given below. Write `async def divide_all(pairs)`. It calls `divide(a, b)` for every pair at the same time. Use `return_exceptions=True`, so that a failing pair gives its exception object in the results instead of stopping the whole batch.

**Sample input and expected output**

```python
results = await divide_all([(4, 2), (1, 0), (9, 3)])
# Expected: [2.0, a ZeroDivisionError object, 3.0]
# The failing division must not discard the two successful results.
await divide_all([])  # Expected: []
```

In [ ]:
import asyncio

async def divide(a, b):
    await asyncio.sleep(0)
    return a / b

async def divide_all(pairs):
    # your code here
    pass

<details>
<summary><b>Show solution</b></summary>

```python
import asyncio

async def divide(a, b):
    await asyncio.sleep(0)
    return a / b

async def divide_all(pairs):
    return await asyncio.gather(*(divide(a, b) for a, b in pairs), return_exceptions=True)
```

</details>

## Exercise 6: a result or `None` after a timeout

**Difficulty:** Medium · about 10 min

Write `async def first_result_or_none(make_coro, seconds)`. `make_coro` is a function that returns a coroutine. Wait for that coroutine for at most `seconds`. Return its result, or `None` when the wait times out.

**Sample input and expected output**

```python
async def fast():
    return 0

async def slow():
    await asyncio.sleep(1)
    return "late"

await first_result_or_none(fast, 0.1)  # Expected: 0
await first_result_or_none(slow, 0.01) # Expected: None
```


*Hint: `asyncio.wait_for` raises `asyncio.TimeoutError` when time runs out.*

In [ ]:
import asyncio

async def first_result_or_none(make_coro, seconds):
    # your code here
    pass

<details>
<summary><b>Show solution</b></summary>

```python
import asyncio

async def first_result_or_none(make_coro, seconds):
    try:
        return await asyncio.wait_for(make_coro(), seconds)
    except asyncio.TimeoutError:
        return None
```

</details>